In [109]:
import pandas as pd

users = pd.read_csv("data/DATA1/users.csv")
orders = pd.read_parquet("data/DATA1/orders.parquet")

# df2 = pd.read_csv("data/DATA2/users.csv")
# df3 = pd.read_csv("data/DATA3/users.csv")

# df = pd.concat([df1, df2, df3])

df = orders.merge(users, left_on='user_id', right_on='id')

# df['paid_price'] = df['quantity'] * df['unit_price']

# print(users.columns)
# print(orders.columns)
print(df)
print(df.columns)


        id_x  user_id  book_id  quantity unit_price  \
0      71389    47288    18976         2     27.00$   
1      66343    47049    19403         1     €50¢50   
2      72606    46685    19500         1  USD 45.99   
3      68462    45336    18992         1    € 71.00   
4      72691    45311    19388         1    52.25 $   
...      ...      ...      ...       ...        ...   
11232  70045    45032    18966         1  57.99 EUR   
11233  72164    46481    18864         2     $71¢00   
11234  70697    44686    19083         5       €50.   
11235  64646    46203    19190         1   44.75USD   
11236  67438    45415    18868         5      28USD   

                          timestamp  \
0            10/01/24 10:38:08 A.M.   
1                 10:14;19-Oct-2024   
2               22:13:35,2025-07-02   
3               2025-10-20 16:25:20   
4      08:48:47 A.M.,28-August-2024   
...                             ...   
11232             04:51:02;04/06/25   
11233      12:38:25 A.M. 20

cleaning

In [110]:
df = df.drop_duplicates()

# correcting data to proper way
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce', utc=True)
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')

# dropped trash (n/a)
df = df.dropna(subset=['timestamp', 'quantity', 'unit_price'])

/var/folders/wn/yv_4s6ys35z6ldk5d54s1k6h0000gn/T/ipykernel_33205/3239800573.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce', utc=True)


In [111]:
# if it's in EUR need to multiply
# df.loc[df['currency'] == 'EUR', 'unit_price'] *= 1.2

# final sum
df['paid_price'] = df['quantity'] * df['unit_price']

In [112]:
import yaml
import pandas as pd

with open("data/DATA1/books.yaml", "r") as file:
    books_data = yaml.safe_load(file)

books = pd.DataFrame(books_data)


In [113]:
books.columns = books.columns.str.replace(':', '')


print(books.head())
print(books.columns)

      id                         title  \
0  19199  The Yellow Meads of Asphodel   
1  19398         From Here to Eternity   
2  19483               Eyeless in Gaza   
3  19506                 Precious Bane   
4  19570                   City of God   

                                             author                    genre  \
0                                     Carolyne West                  Classic   
1  Rep. Heath Stiedemann, Gino Welch, Haydee Larson              Short story   
2                                    Vannessa Price  Biography/Autobiography   
3                                   Miss Yong Wyman        Realistic fiction   
4                                      Travis Moore        Suspense/Thriller   

                 publisher  year  
0    Mainstream Publishing  2009  
1            Vintage Books  2001  
2           Pavilion Books  1886  
3      New English Library  2021  
4  Bellevue Literary Press  1847  
Index(['id', 'title', 'author', 'genre', 'publisher', 'y

In [114]:
df = df.merge(books, left_on='book_id', right_on='id')

In [115]:
df['author_set'] = df['author'].apply(
    lambda x: tuple(sorted([a.strip() for a in x.split(',')]))
)

In [116]:
unique_author_sets = df['author_set'].nunique()

print(unique_author_sets)

0


In [117]:
popular_authors = (
    df.groupby('author_set')['quantity']
    .sum()
    .sort_values(ascending=False)
)

print(popular_authors.head(1))

Series([], Name: quantity, dtype: int32)


In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce', utc=True)

df['date'] = df['timestamp'].dt.date


daily_revenue = df.groupby('date')['paid_price'].sum()
top5_days = daily_revenue.sort_values(ascending=False).head(5)

print(df.columns)

Index(['id_x', 'user_id', 'book_id', 'quantity', 'unit_price', 'timestamp',
       'shipping', 'id_y', 'name', 'address', 'phone', 'email', 'paid_price',
       'id', 'title', 'author', 'genre', 'publisher', 'year', 'author_set',
       'date'],
      dtype='str')
